In [1]:
import sys
sys.path.append("src/")

from pathlib import Path
import tqdm 
import random
import time
from collections import defaultdict
import matplotlib.pyplot as plt

from proxyrecord import IpRecord
from utils import filecache

In [7]:
OUTPUT_FILE = Path("data/rpaas-data-enriched.json")
INPUT_FILE = Path("/DATASETS/ip_captured_as_web_proxy.tsv")

In [10]:
def iter_ip_chunks(path, chunk_size=100000):
    """
    Generator that yields lists of IPs from a TSV file in chunks.

    - Skips comment lines starting with '#'
    - Assumes IP is the first column, separated by whitespace (tabs or spaces)
    """
    chunk = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line or line.startswith("#"):
                continue 

            parts = line.split()
            if not parts:
                continue

            ip = parts[0]
            chunk.append(ip)

            if len(chunk) >= chunk_size:
                yield chunk
                chunk = []

    if chunk:
        yield chunk

In [23]:
total_count = 0
results = dict()

for i, ip_chunk in enumerate(iter_ip_chunks(INPUT_FILE, chunk_size=10000)):
    for ip in ip_chunk:
        record = IpRecord(ip, False)
        record.load_whois()
        if "asn" in record.whois and record.whois["asn"] == 1136:
            record.is_kpn = True
            results[record.ip] = record
        
    total_count += len(ip_chunk)

print(f"We saw {total_count} unique IP's")
print(f"We saw {len(results)} KPN IP's")

We saw 6419987 unique IP's
We saw 12037 KPN IP's


In [24]:
filecache.write_list_to_file(OUTPUT_FILE, list(results.values()))